# Phase 3, experiment 12: averaging GNNs trained with different seeds

**Why this experiment:** Blending the GNN from notebook 13 with LightGBM raised the public leaderboard
score from 0.94660 (LightGBM alone) to 0.95037 (GNN weight 0.1) and 0.95325 (GNN weight 0.2), even
though the validation folds called the blend level. That GNN was trained once, with one random seed.
A neural network's result depends on its random starting weights, so this notebook checks:

1. **How much does the GNN change from one seed to the next?** If a lot, part of the leaderboard gain
   may be luck.
2. **Is the average of several GNNs better?** Averaging networks trained from different starting
   points usually gives steadier and slightly better predictions.

**The one change in this experiment:** the same GNN as notebook 13 is trained 5 times per fold, with
seeds 42 to 46, and the 5 predicted probabilities are averaged.

**Other change (decided 14 Oct 2026):** 100 epochs instead of 200, because in notebook 13 the
validation AUC stopped improving after about 25 epochs. This halves the run time.

**Decision:** carry on with the GNN (seed averaging) before finalising.

As in notebook 13, LightGBM is not used here because it crashes in the same session as PyTorch on
this Mac. Predictions are saved to `../outputs/` for `16_gnn_seeds_blend.ipynb`.

Run time: about 15 to 20 minutes (5 seeds x 4 trainings: three folds and the final model).

## 1. Load and build the graph (same as notebook 13)

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score

DATA_DIR = "../data"
OUT_DIR = "../outputs"
os.makedirs(OUT_DIR, exist_ok=True)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
raw_cols = [c for c in train.columns if c.startswith("feat_")]

# One table of nodes: every train and test transaction. Test rows have no label (NaN).
nodes = pd.concat([train[["txId", "time_step", "label"] + raw_cols],
                   test[["txId", "time_step"] + raw_cols]], ignore_index=True)

# PyTorch Geometric refers to nodes by row number, not by txId
row_of = pd.Series(np.arange(len(nodes)), index=nodes["txId"])

# Keep only edges whose two ends are both in train or test
known = edges["txId1"].isin(row_of.index) & edges["txId2"].isin(row_of.index)
kept = edges[known]
edge_index = torch.tensor(np.vstack([row_of[kept["txId1"]].values, row_of[kept["txId2"]].values]),
                          dtype=torch.long)

time_step = nodes["time_step"].values
label = nodes["label"].values          # NaN for unlabeled and test rows
print("nodes:", len(nodes), " edges kept:", edge_index.shape[1])

/Users/dorislai/Downloads/bt4012_kaggle_competition/bt4012-kaggle/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


nodes: 157101  edges kept: 175918


## 2. The model (same as notebook 13)

In [2]:
class DirectedSAGE(torch.nn.Module):
    # Two GraphSAGE layers. Each layer adds a predecessor part and a successor part.

    def __init__(self, n_features, hidden):
        super().__init__()
        self.pred1 = SAGEConv(n_features, hidden)
        self.succ1 = SAGEConv(n_features, hidden)
        self.pred2 = SAGEConv(hidden, hidden)
        self.succ2 = SAGEConv(hidden, hidden)
        self.out = torch.nn.Linear(hidden, 1)

    def forward(self, x, edge_index):
        reversed_edges = edge_index.flip(0)   # reversed edges point from successors to the node
        h = F.relu(self.pred1(x, edge_index) + self.succ1(x, reversed_edges))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.pred2(h, edge_index) + self.succ2(h, reversed_edges))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.out(h).squeeze(-1)

## 3. Training function

Same as notebook 13, except that the seed is now an argument, and the features are scaled in a separate
function so the scaling is done once per period and shared by all 5 seeds.

In [3]:
EPOCHS = 100
HIDDEN = 64
SEEDS = [42, 43, 44, 45, 46]

def scaled_features(train_last_step):
    # Scale features with the mean and std of the training-period rows only, clip to -10..10
    in_train_period = time_step <= train_last_step
    mean = nodes.loc[in_train_period, raw_cols].mean()
    std = nodes.loc[in_train_period, raw_cols].std().replace(0, 1)
    scaled = ((nodes[raw_cols] - mean) / std).clip(-10, 10)
    return torch.tensor(scaled.values, dtype=torch.float32)

def train_gnn(x, train_last_step, seed):
    # Train one GNN on labeled rows with time_step <= train_last_step; return a probability per node
    train_rows = np.where((time_step <= train_last_step) & ~np.isnan(label))[0]
    y = torch.tensor(label[train_rows], dtype=torch.float32)
    pos_weight = (y == 0).sum() / (y == 1).sum()

    torch.manual_seed(seed)                       # this seed decides the starting weights and dropout
    model = DirectedSAGE(len(raw_cols), HIDDEN)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        logits = model(x, edge_index)
        loss = F.binary_cross_entropy_with_logits(logits[train_rows], y, pos_weight=pos_weight)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(x, edge_index)).numpy()

## 4. Train 5 seeds on each fold

For every fold, each seed's AUC is recorded, then the 5 probabilities are averaged and the average is
scored.

In [4]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]

auc_rows = []           # one row per (fold, seed), plus the average of the seeds
fold_predictions = []   # averaged GNN predictions for the labeled validation rows

for fold in FOLDS:
    x = scaled_features(fold["train_last"])
    in_valid = (time_step > fold["train_last"]) & (time_step <= fold["valid_last"]) & ~np.isnan(label)
    valid_rows = np.where(in_valid)[0]
    valid_labels = label[valid_rows]

    seed_probs = []
    for seed in SEEDS:
        probs = train_gnn(x, fold["train_last"], seed)
        seed_probs.append(probs)
        auc_rows.append({"fold": fold["name"], "model": f"seed {seed}",
                         "auc": roc_auc_score(valid_labels, probs[valid_rows])})
        print("done:", fold["name"], "seed", seed)

    # Average the probabilities of all seeds for every node
    avg_probs = np.mean(seed_probs, axis=0)
    auc_rows.append({"fold": fold["name"], "model": "average of seeds",
                     "auc": roc_auc_score(valid_labels, avg_probs[valid_rows])})

    fold_predictions.append(pd.DataFrame({
        "fold": fold["name"],
        "txId": nodes["txId"].values[valid_rows],
        "gnn_score": avg_probs[valid_rows],
    }))

pd.concat(fold_predictions).to_csv(f"{OUT_DIR}/gnn5_fold_predictions.csv", index=False)

done: fold 1 seed 42
done: fold 1 seed 43
done: fold 1 seed 44
done: fold 1 seed 45
done: fold 1 seed 46
done: fold 2 seed 42
done: fold 2 seed 43
done: fold 2 seed 44
done: fold 2 seed 45
done: fold 2 seed 46
done: fold 3 seed 42
done: fold 3 seed 43
done: fold 3 seed 44
done: fold 3 seed 45
done: fold 3 seed 46


## 5. Results

In [ ]:
fold_names = [f["name"] for f in FOLDS]
model_order = [f"seed {s}" for s in SEEDS] + ["average of seeds"]

auc_table = pd.DataFrame(auc_rows).pivot(index="model", columns="fold", values="auc").loc[model_order]
auc_table["mean"] = auc_table[fold_names].mean(axis=1)

single = auc_table.loc[[f"seed {s}" for s in SEEDS]]
print("Range over the single seeds (max - min):")
print((single.max() - single.min()).round(4))

auc_table.round(4)

Range over the single seeds (max - min):
fold
fold 1    0.0116
fold 2    0.0062
fold 3    0.0075
mean      0.0027
dtype: float64


fold,fold 1,fold 2,fold 3,mean
model,,,,
seed 42,0.9533,0.9113,0.9803,0.9483
seed 43,0.9597,0.9113,0.9780,0.9497
seed 44,0.9533,0.9111,0.9833,0.9492
seed 45,0.9633,0.9052,0.9758,0.9481
seed 46,0.9517,0.9104,0.9787,0.9469
average of seeds,0.9615,0.9122,0.9857,0.9531


## 6. Train 5 seeds on all labeled rows and predict the test set

In [6]:
x_final = scaled_features(35)
final_probs = []
for seed in SEEDS:
    final_probs.append(train_gnn(x_final, 35, seed))
    print("done: final model, seed", seed)
avg_final = np.mean(final_probs, axis=0)

is_test = np.isin(nodes["txId"].values, test["txId"].values)
pd.DataFrame({"txId": nodes["txId"].values[is_test], "gnn_score": avg_final[is_test]}) \
    .to_csv(f"{OUT_DIR}/gnn5_test_predictions.csv", index=False)
print("Saved averaged test predictions for", is_test.sum(), "test transactions")

done: final model, seed 42
done: final model, seed 43
done: final model, seed 44
done: final model, seed 45
done: final model, seed 46
Saved averaged test predictions for 15329 test transactions


## Findings

- **How much does a single GNN vary between seeds?** On a single fold the 5 seeds differ by up to 0.0116
  (fold 1: 0.9517 to 0.9633), 0.0062 (fold 2: 0.9052 to 0.9113) and 0.0075 (fold 3: 0.9758 to 0.9833).
  Their fold means are closer, 0.9469 to 0.9497 (a range of 0.0027), because a seed that does well on
  one fold often does less well on another. So a single GNN's score on one period can move by about
  0.01 just from its random starting weights.
- **Is the average of the 5 seeds better than a single GNN?** Yes. Its fold mean (0.9531) is above every
  single seed (0.9469 to 0.9497). It is the best of all six rows on fold 2 (0.9122) and fold 3 (0.9857);
  on fold 1 (0.9615) only seed 45 (0.9633) is higher.
- **How does it compare with the single GNN from notebook 13 (0.9562 / 0.9065 / 0.9807, mean 0.9478)?**
  The average is better on every fold (+0.0053, +0.0057 and +0.0050), and its mean is 0.0053 higher.
  Seed 42 at 100 epochs (mean 0.9483) is almost the same as notebook 13's seed 42 at 200 epochs
  (0.9478), which supports halving the epochs. The averaged GNN is still well below LightGBM on its own
  (fold mean 0.9808); its value is as a blend partner (notebook 16).
